# Задание

## Изучите данные и выполните следующие задачи. Выполните задание с помощью SQL.


Есть 3 таблицы с данными. В таблице *client_info* хранится информация о клиентах, в частности, пол, возраст и тд. В таблице payments хранится информация о всех клиентских платежах.<br>

В таблице *city_info* хранится расшифровка идентификаторов городов и наименования федеральных округов, в которых расположены клиентские города.<br>

📁 **Таблица payments**<br>

* id_client (уникальный идентификатор клиента)<br>
* time_payment (дата и время платежа в формате "гггг-мм-дд чч:мм:сс")<br>
* amt_payment (размер платежа)<br>

📁 **Таблица client_info**<br>

* id_client (уникальный идентификатор клиента)<br>
* gender (пол клиента)<br>
* age (возраст клиента)<br>
* id_city (идентификатор города клиента)<br>

📁 **Таблица city_info**

* id_city (идентификатор города клиента)<br>
* name_city (название города клиента)<br>
* name_region (наименование федерального округа, в котором расположен данный город)<br>

**Задание 2.1**

*Для федерального округа "Поволжье" выведите динамику суммарных платежей по дням.*<br>

```sql

WITH tabl1 as
(SELECT *
FROM client_info as client
JOIN city_info as city
 on client.id_city = city.id_city
)
select date_trunc('day', to_Timestamp(time_payment,'DD.MM.YY HH24:MI')) as dd
	  ,sum(amt_payment) as sum_payment
FROM payments as pp
JOIN tabl1 as new_tt
	on pp.id_client = new_tt.id_client
WHERE new_tt.name_region = 'Поволжский федеральный округ'
GROUP by dd

```
**Задание 2.2**

*Для каждого города найдите долю мужчин (% мужчин среди всех клиентов в данном городе). Ограничьтесь только клиентами, которым от 20 до 40 лет. В выводе используйте названия городов, а не идентификаторы.*<br>

```sql

SELECT city.name_city
	  ,sum(case when client.gender = 'M' then 1.0 else 0.0 end) / count(id_client) * 100 as share_man
FROM client_info as client
join city_info as city
  on client.id_city = city.id_city
where client.age BETWEEN 20 and 40
GROUP by city.name_city
```


**Задание 2.3**

*Определите средний возраст по тем клиентам, которые ни разу ничего не заплатили.*<br>

```sql

SELECT avg(age)
FROM client_info as client
left JOIN payments as pp
  on client.id_client = pp.id_client
WHERE amt_payment is Null
```

 
**Задание 2.4**

*Для каждого федерального округа выделите первые три платежа.*<br>

```sql

with tabl1 as 
(select name_region 
	,time_payment
	,amt_payment 
	,ROW_NUMBER() OVER (PARTITION BY name_region ORDER BY  time_payment)  as row_num
	 from payments as pp
	 join client_info as inform 
		on pp.id_client = inform.id_client
	 join city_info as city
		on inform.id_city  = city.id_city
	)
select name_region 
	,amt_payment
from tabl1
where row_num <= 3
```


**Задание 2.5**

*Ограничьтесь клиентами из федеральных округов "Южный" и "Северный". Для каждого города рассчитайте, сколько в среднем времени проходит между платежами одного клиента.*<br>

```sql

with tabl1 as
(select pp.id_client 
  	   ,name_region 
	   ,name_city 
       ,time_payment 
from payments as pp
	 join client_info as inform 
		on pp.id_client = inform.id_client
	 join city_info as city
		on inform.id_city  = city.id_city
where name_region = 'Южный федеральный округ' or  name_region = 'Северный федеральный округ'
),
tabl2 as
(select	name_region 
		,name_city 
		,time_payment 
		,LEAD (time_payment) OVER (PARTITION BY name_city, id_client ORDER BY time_payment) as next_time
from tabl1
)

select name_city
	,avg(time_payment - next_time) as avg_time
from tabl2
GROUP by tabl2.name_city
```